In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

**Layer Normalization Placement (Pre-Norm vs Post-Norm)**

In [2]:
class PreNormTransformerLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward, dropout):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
            nn.Dropout(dropout)
        )
    
    def forward(self, x, mask=None):
        x = x + self.self_attn(self.norm1(x), self.norm1(x), self.norm1(x), attn_mask=mask)[0]
        x = x + self.ffn(self.norm2(x))
        return x

**GELU Activation (vs ReLU)**

`GELU is smoother and often better for transformers`

In [3]:
activation = nn.GELU()  # Gaussian Error Linear Unit

**RMSNorm (Root Mean Square Layer Normalization)**

In [4]:
class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-8):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(d_model))
        self.eps = eps
    
    def forward(self, x):
        norm = x.norm(dim=-1, keepdim=True) * (x.shape[-1] ** -0.5)
        return self.scale * x / (norm + self.eps)

**SwiGLU Activation**

In [ ]:
class SwiGLU(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.linear = nn.Linear(dim, dim * 2)
    
    def forward(self, x):
        x, gate = self.linear(x).chunk(2, dim=-1)
        return F.silu(x) * gate